# Baseline Models Comparison

This notebook compares three baseline approaches for intent classification:
1. TF-IDF + Logistic Regression (sklearn)
2. Sentence Embeddings (all-MiniLM-L6-v2) + Logistic Regression
3. Zero-shot with LLM API (Claude Haiku / GPT-4o-mini)

Metrics logged to wandb: Accuracy, Macro-F1, Per-class F1

## Setup and Dependencies

In [1]:
# Install required libraries
!pip install datasets transformers scikit-learn sentence-transformers wandb anthropic openai -q

In [2]:
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
WANDB_API_KEY = userdata.get('WANDB_API_KEY')

In [3]:
import os
import time
import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

import wandb
from anthropic import Anthropic

In [4]:
# Initialize wandb
wandb.init(
    project="intent-classification",
    name="baseline-models-comparison",
    config={
        "tfidf_model": "sklearn-tfidf-logistic",
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
        "zeroshot_model": "claude-haiku",
    }
)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sophieb to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [anthropic] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


## Load and Prepare Dataset

In [5]:
# Load the dataset
dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
print(f"Dataset splits: {dataset.keys()}")
print(f"Train set size: {len(dataset['train'])}")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset splits: dict_keys(['train'])
Train set size: 26872


In [6]:
# Use same split as test_hf_pipeline.ipynb
dataset['train'] = dataset['train'].shuffle(seed=42)

train_testvalid = dataset['train'].train_test_split(test_size=0.3, seed=42)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

split_dataset = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'validation': test_valid['train']
})

print(f"Train set size: {len(split_dataset['train'])}")
print(f"Validation set size: {len(split_dataset['validation'])}")
print(f"Test set size: {len(split_dataset['test'])}")

Train set size: 18810
Validation set size: 4031
Test set size: 4031


In [7]:
# Get label information
labels = split_dataset['train']['intent']
unique_labels = sorted(set(labels))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print(f"Number of classes: {len(unique_labels)}")
print(f"\nClasses: {unique_labels}")

Number of classes: 27

Classes: ['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']


In [8]:
# Prepare data for training
X_train = split_dataset['train']['instruction']
y_train = np.array([label2id[label] for label in split_dataset['train']['intent']])

X_test = split_dataset['test']['instruction']
y_test = np.array([label2id[label] for label in split_dataset['test']['intent']])

X_val = split_dataset['validation']['instruction']
y_val = np.array([label2id[label] for label in split_dataset['validation']['intent']])

print(f"Training data shape: {len(X_train)}")
print(f"Test data shape: {len(X_test)}")
print(f"Validation data shape: {len(X_val)}")

Training data shape: 18810
Test data shape: 4031
Validation data shape: 4031


## Model 1: TF-IDF + Logistic Regression

In [9]:
print("="*60)
print("MODEL 1: TF-IDF + Logistic Regression")
print("="*60)

# Training
start_time = time.time()
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train_tfidf, y_train)
train_time = time.time() - start_time

print(f"\nTraining time: {train_time:.2f}s")

# Prediction
start_time = time.time()
y_pred_lr = lr_model.predict(X_test_tfidf)
inference_time = time.time() - start_time
inference_time_per_sample = (inference_time / len(X_test)) * 1000  # ms per sample

print(f"\nInference time: {inference_time_per_sample:.2f}s")

# Metrics
accuracy_lr = accuracy_score(y_test, y_pred_lr)
macro_f1_lr = f1_score(y_test, y_pred_lr, average='weighted')
per_class_f1_lr = f1_score(y_test, y_pred_lr, average=None)

print(f"\nAccuracy: {accuracy_lr:.4f}")
print(f"Macro F1: {macro_f1_lr:.4f}")
print(f"Inference time per sample: {inference_time_per_sample:.2f}ms")
print(f"Cost per 1k predictions: ~$0 (local inference)")
print(f"\nPer-class F1 Scores:")
for label, f1 in zip(unique_labels, per_class_f1_lr):
    print(f"  {label}: {f1:.4f}")

# Log to wandb
wandb.log({
    "tfidf_accuracy": accuracy_lr,
    "tfidf_macro_f1": macro_f1_lr,
    "tfidf_inference_time_ms": inference_time_per_sample,
    "tfidf_cost_per_1k": 0.0,
    **{f"tfidf_f1_{label}": f1 for label, f1 in zip(unique_labels, per_class_f1_lr)}
})

MODEL 1: TF-IDF + Logistic Regression

Training time: 5.67s

Inference time: 0.00s

Accuracy: 0.9913
Macro F1: 0.9913
Inference time per sample: 0.00ms
Cost per 1k predictions: ~$0 (local inference)

Per-class F1 Scores:
  cancel_order: 0.9868
  change_order: 0.9739
  change_shipping_address: 0.9927
  check_cancellation_fee: 0.9925
  check_invoice: 0.9766
  check_payment_methods: 1.0000
  check_refund_policy: 0.9940
  complaint: 1.0000
  contact_customer_service: 0.9862
  contact_human_agent: 0.9836
  create_account: 0.9850
  delete_account: 0.9867
  delivery_options: 0.9962
  delivery_period: 0.9967
  edit_account: 0.9967
  get_invoice: 0.9817
  get_refund: 0.9937
  newsletter_subscription: 1.0000
  payment_issue: 0.9826
  place_order: 0.9930
  recover_password: 1.0000
  registration_problems: 0.9843
  review: 1.0000
  set_up_shipping_address: 0.9929
  switch_account: 1.0000
  track_order: 0.9932
  track_refund: 1.0000


## Model 2: Sentence Embeddings + Logistic Regression

In [10]:
print("\n" + "="*60)
print("MODEL 2: Sentence Embeddings + Logistic Regression")
print("="*60)

# Load embedding model
print("\nLoading embedding model...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Encode texts
print("Encoding training texts...")
start_time = time.time()
X_train_emb = embedding_model.encode(list(X_train), batch_size=32, show_progress_bar=True)
X_test_emb = embedding_model.encode(list(X_test), batch_size=32, show_progress_bar=True)
encoding_time = time.time() - start_time

print(f"Encoding time: {encoding_time:.2f}s")

# Train logistic regression on embeddings
print("\nTraining Logistic Regression...")
start_time = time.time()
lr_emb_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_emb_model.fit(X_train_emb, y_train)
train_time = time.time() - start_time

print(f"Training time: {train_time:.2f}s")

# Prediction
start_time = time.time()
y_pred_emb = lr_emb_model.predict(X_test_emb)
inference_time = time.time() - start_time
inference_time_per_sample = (inference_time / len(X_test)) * 1000  # ms per sample

# Metrics
accuracy_emb = accuracy_score(y_test, y_pred_emb)
macro_f1_emb = f1_score(y_test, y_pred_emb, average='weighted')
per_class_f1_emb = f1_score(y_test, y_pred_emb, average=None)

print(f"\nAccuracy: {accuracy_emb:.4f}")
print(f"Macro F1: {macro_f1_emb:.4f}")
print(f"Inference time per sample: {inference_time_per_sample:.2f}ms")
print(f"Cost per 1k predictions: ~$0 (local inference)")
print(f"\nPer-class F1 Scores:")
for label, f1 in zip(unique_labels, per_class_f1_emb):
    print(f"  {label}: {f1:.4f}")

# Log to wandb
wandb.log({
    "embedding_accuracy": accuracy_emb,
    "embedding_macro_f1": macro_f1_emb,
    "embedding_inference_time_ms": inference_time_per_sample,
    "embedding_cost_per_1k": 0.0,
    **{f"embedding_f1_{label}": f1 for label, f1 in zip(unique_labels, per_class_f1_emb)}
})


MODEL 2: Sentence Embeddings + Logistic Regression

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding training texts...


Batches:   0%|          | 0/588 [00:00<?, ?it/s]

Batches:   0%|          | 0/126 [00:00<?, ?it/s]

Encoding time: 14.64s

Training Logistic Regression...
Training time: 2.73s

Accuracy: 0.9943
Macro F1: 0.9943
Inference time per sample: 0.00ms
Cost per 1k predictions: ~$0 (local inference)

Per-class F1 Scores:
  cancel_order: 0.9902
  change_order: 0.9900
  change_shipping_address: 0.9963
  check_cancellation_fee: 1.0000
  check_invoice: 0.9864
  check_payment_methods: 1.0000
  check_refund_policy: 0.9941
  complaint: 1.0000
  contact_customer_service: 0.9966
  contact_human_agent: 0.9967
  create_account: 0.9970
  delete_account: 1.0000
  delivery_options: 0.9962
  delivery_period: 0.9967
  edit_account: 1.0000
  get_invoice: 0.9880
  get_refund: 0.9809
  newsletter_subscription: 1.0000
  payment_issue: 0.9931
  place_order: 0.9965
  recover_password: 1.0000
  registration_problems: 0.9906
  review: 0.9968
  set_up_shipping_address: 0.9930
  switch_account: 1.0000
  track_order: 0.9866
  track_refund: 0.9842


## Model 3: Zero-shot with LLM API (Claude Haiku)

In [12]:
print("\n" + "="*60)
print("MODEL 3: Zero-shot Classification with Claude Haiku")
print("="*60)

# Initialize Anthropic client
client = Anthropic(api_key=ANTHROPIC_API_KEY)

# Use subset of test set for cost efficiency
test_sample_size = min(200, len(X_test))
X_test_sample = X_test[:test_sample_size]
y_test_sample = y_test[:test_sample_size]

print(f"\nUsing {test_sample_size} test samples for zero-shot evaluation")

# Create prompt template
intent_list = ", ".join(unique_labels)

# Classify samples
print("\nCalling Claude Haiku API...")
start_time = time.time()
y_pred_claude = []
total_input_tokens = 0
total_output_tokens = 0

for i, text in enumerate(X_test_sample):
    if i % 50 == 0:
        print(f"  Processed {i}/{test_sample_size} samples")

    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=50,
        messages=[
            {
                "role": "user",
                "content": f"""Classify the following customer support query into ONE of these intent categories:
{intent_list}

Query: {text}

Respond with ONLY the intent category name, nothing else."""
            }
        ]
    )

    response_text = message.content[0].text.strip().lower()
    # Find matching intent
    pred_intent = unique_labels[0]
    for intent in unique_labels:
        if intent.lower() in response_text:
            pred_intent = intent
            break

    y_pred_claude.append(label2id[pred_intent])
    total_input_tokens += message.usage.input_tokens
    total_output_tokens += message.usage.output_tokens

inference_time = time.time() - start_time
y_pred_claude = np.array(y_pred_claude)

# Calculate costs
cost_per_input_1k = (0.80 / 1_000_000) * 1000
cost_per_output_1k = (4.00 / 1_000_000) * 1000
total_cost = (total_input_tokens * 0.80 / 1_000_000) + (total_output_tokens * 4.00 / 1_000_000)
cost_per_1k = ((total_input_tokens * 0.80 + total_output_tokens * 4.00) / 1_000_000) * 1000 / test_sample_size

inference_time_per_sample = (inference_time / test_sample_size) * 1000  # ms per sample

# Metrics
accuracy_claude = accuracy_score(y_test_sample, y_pred_claude)
macro_f1_claude = f1_score(y_test_sample, y_pred_claude, average='weighted')
per_class_f1_claude = f1_score(y_test_sample, y_pred_claude, average=None)

print(f"\nAccuracy: {accuracy_claude:.4f}")
print(f"Macro F1: {macro_f1_claude:.4f}")
print(f"Inference time per sample: {inference_time_per_sample:.2f}ms")
print(f"\nToken Usage:")
print(f"  Total input tokens: {total_input_tokens:,}")
print(f"  Total output tokens: {total_output_tokens:,}")
print(f"  Total cost: ${total_cost:.4f}")
print(f"  Cost per 1k predictions: ${cost_per_1k:.4f}")
print(f"\nPer-class F1 Scores:")
for label, f1 in zip(unique_labels, per_class_f1_claude):
    print(f"  {label}: {f1:.4f}")

# Log to wandb
wandb.log({
    "claude_accuracy": accuracy_claude,
    "claude_macro_f1": macro_f1_claude,
    "claude_inference_time_ms": inference_time_per_sample,
    "claude_cost_per_1k": cost_per_1k,
    "claude_total_cost": total_cost,
    "claude_input_tokens": total_input_tokens,
    "claude_output_tokens": total_output_tokens,
    **{f"claude_f1_{label}": f1 for label, f1 in zip(unique_labels, per_class_f1_claude)}
})


MODEL 3: Zero-shot Classification with Claude Haiku

Using 200 test samples for zero-shot evaluation

Calling Claude Haiku API...
  Processed 0/200 samples
  Processed 50/200 samples
  Processed 100/200 samples
  Processed 150/200 samples

Accuracy: 0.8650
Macro F1: 0.8573
Inference time per sample: 592.91ms

Token Usage:
  Total input tokens: 34,441
  Total output tokens: 1,347
  Total cost: $0.0329
  Cost per 1k predictions: $0.1647

Per-class F1 Scores:
  cancel_order: 0.8571
  change_order: 0.9091
  change_shipping_address: 0.8571
  check_cancellation_fee: 1.0000
  check_invoice: 0.8571
  check_payment_methods: 1.0000
  check_refund_policy: 0.9091
  complaint: 0.7619
  contact_customer_service: 0.7200
  contact_human_agent: 0.8333
  create_account: 1.0000
  delete_account: 1.0000
  delivery_options: 0.9091
  delivery_period: 0.8000
  edit_account: 0.7500
  get_invoice: 0.9630
  get_refund: 0.7143
  newsletter_subscription: 1.0000
  payment_issue: 1.0000
  place_order: 0.8571
  rec

## Results Comparison

In [13]:
# Create comparison table
results_df = pd.DataFrame({
    'Model': ['TF-IDF + LR', 'Embeddings + LR', 'Claude Haiku (Zero-shot)'],
    'Accuracy': [accuracy_lr, accuracy_emb, accuracy_claude],
    'Macro F1': [macro_f1_lr, macro_f1_emb, macro_f1_claude],
    'Inference Time (ms)': [
        inference_time_per_sample,
        inference_time_per_sample,
        (inference_time / test_sample_size) * 1000
    ],
    'Cost per 1k': ['$0.00', '$0.00', f'${cost_per_1k:.4f}']
})

print("\n" + "="*80)
print("FINAL RESULTS COMPARISON")
print("="*80)
print(results_df.to_string(index=False))

# Log comparison to wandb
wandb.log({"results_comparison": wandb.Table(dataframe=results_df)})


FINAL RESULTS COMPARISON
                   Model  Accuracy  Macro F1  Inference Time (ms) Cost per 1k
             TF-IDF + LR  0.991317  0.991320           592.910029       $0.00
         Embeddings + LR  0.994294  0.994291           592.910029       $0.00
Claude Haiku (Zero-shot)  0.865000  0.857269           592.910029     $0.1647


In [14]:
# Summary and recommendations
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"""
Best Accuracy: {results_df.loc[results_df['Accuracy'].idxmax(), 'Model']}
Best Macro F1: {results_df.loc[results_df['Macro F1'].idxmax(), 'Model']}
Fastest Inference: {results_df.loc[results_df['Inference Time (ms)'].idxmin(), 'Model']}
Lowest Cost: {results_df.loc[results_df['Cost per 1k'].idxmin(), 'Model']}

Recommendations:
- For production use with cost constraints: Use Embeddings + LR (good accuracy, zero cost)
- For best accuracy: Compare TF-IDF + LR and Embeddings + LR performance
- For real-time requirements: Both local models (TF-IDF and Embeddings) are sub-ms per sample
- Zero-shot LLM approach: Good for few-shot scenarios but higher latency and cost
""")

wandb.finish()


SUMMARY

Best Accuracy: Embeddings + LR
Best Macro F1: Embeddings + LR
Fastest Inference: TF-IDF + LR
Lowest Cost: TF-IDF + LR

Recommendations:
- For production use with cost constraints: Use Embeddings + LR (good accuracy, zero cost)
- For best accuracy: Compare TF-IDF + LR and Embeddings + LR performance
- For real-time requirements: Both local models (TF-IDF and Embeddings) are sub-ms per sample
- Zero-shot LLM approach: Good for few-shot scenarios but higher latency and cost



claude_accuracy,▁
claude_cost_per_1k,▁
claude_f1_cancel_order,▁
claude_f1_change_order,▁
claude_f1_change_shipping_address,▁
claude_f1_check_cancellation_fee,▁
claude_f1_check_invoice,▁
claude_f1_check_payment_methods,▁
claude_f1_check_refund_policy,▁
claude_f1_complaint,▁
+86,...
